# Phenotype query + filtering check

Smoke test for the querying/filtering half of `02_phenotype`, run against the
real CDR before trusting the full `residualize_phenotypes.ipynb` pipeline.
Uses a short, high-confidence subset of `docs/phenotype_list.tsv` (the
confirmed anthropometric concept_ids only, no `UNCONFIRMED` lipid rows) to
check:

1. BigQuery connectivity under Workbench 2.0
2. the phenotype concept_ids resolve to the expected concepts
3. the measurement pull shape (rows, distinct persons, NA rate)
4. the round-2 keep-list intersection funnel
5. sex_at_birth / age join
6. value ranges are physiologically plausible
7. the residualization step itself (`residualize_lib.R`, base + base_pcs covariate sets) runs cleanly on real data
8. covariate loadings, encoding, and phenotype distributions look sane (`base_pcs_ses`/`base_pcs_zip3_ses` coefficients, SD/mean of PCs, zip3 factor levels, skewness)

**Every cell below prints aggregate/summary statistics only — counts, rates,
ranges — never a person-level row.** That's deliberate, not just for output
hygiene: it keeps this notebook safe to leave un-cleared if the outputs are
ever glanced at, on top of (not instead of) the repo-wide rule to clear
outputs / run `nbstripout` before committing anything run against real AoU
data (see root `README.md`).

## Compute resource

This notebook only issues small aggregate BigQuery queries and pulls a
handful of summary numbers back locally — BigQuery does the heavy lifting
server-side. Workbench 2.0's default cloud environment app size (2 CPU / 13
GB) is enough; no need to size up for this notebook. Reach for a larger
machine only for `residualize_phenotypes.ipynb` (holds the full cohort in
memory) or the GRM stages.

In [ ]:
required_pkgs <- c("dplyr", "readr", "stringr", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs, lib = user_lib)

library(dplyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)
source("../../scripts/local/residualize_lib.R")   # residualize_phenotype(), shared with residualize_phenotypes.ipynb


## Connecting to the CDR

`allofus::aou_connect()` / `aou_sql()` — confirmed working on Workbench 2.0
in practice, same connection pattern `residualize_phenotypes.ipynb` uses.
`aou_sql()` interprets `{CDR}` in the query string via `glue`, resolved to
the workspace's CDR BigQuery dataset automatically — no manual
project/dataset/billing wiring needed.

In [ ]:
con <- aou_connect()   # BigQuery connection to the CDR

run_query <- function(sql) collect(aou_sql(sql))   # aou_sql() returns a lazy tbl; collect() materializes it

## Inputs

- Keep-list: round 2's ancestry-filtering output (`round2_filter.ipynb`,
  1000G-fit ellipsoid) is the primary/default. `round2b` (`reverse_pca_aou.ipynb`,
  AoU-fit ellipsoid, provisional cross-check) is also read in, so both are on
  hand for comparison -- flip `KEEP_LIST_PATH` to `KEEP_LIST_PATH_ROUND2B` to
  run this smoke test against the alternative filtering instead.
- PC covariates: round 2b's PC projections (`reverse_pca_aou.ipynb`'s
  `PC_COVARIATE_PATH` output) -- used in Step 5's `base_pcs` covariate set.
- Phenotype subset: the confirmed rows of `docs/phenotype_list.tsv`, first 3
  — enough to check the query/filter logic without pulling the full panel.

In [ ]:
WORKSPACE_BUCKET <- path.expand("~/workspace/Data from All of Us Controlled Tier /shared-env-pilot")
ANCESTRY_BUCKET_DIR <- file.path(WORKSPACE_BUCKET, "data", "01_ancestry_filtering")

# round 2: 1000G-fit ellipsoid (round2_filter.ipynb)
ROUND2_OUT_DIR <- file.path(ANCESTRY_BUCKET_DIR, "round2_pca")
KEEP_LIST_PATH_ROUND2 <- file.path(ROUND2_OUT_DIR, "round2_keep_ids_ceu_gbr_r2_p99.9999.txt")

# round 2b: AoU-fit ellipsoid, provisional cross-check (reverse_pca_aou.ipynb)
ROUND2B_OUT_DIR <- file.path(ANCESTRY_BUCKET_DIR, "reverse_pca_aou")
KEEP_LIST_PATH_ROUND2B <- file.path(ROUND2B_OUT_DIR, "round2b_keep_ids_aou_fit_r2b_p99.9999.txt")

# round 2b's PC1-PC20 for its retained set (reverse_pca_aou.ipynb's PC_COVARIATE_PATH output)
PC_COVARIATE_PATH <- file.path(ROUND2B_OUT_DIR, "round2b_pc_covariates_p99.9999.txt")

KEEP_LIST_PATH <- KEEP_LIST_PATH_ROUND2   # which keep-list downstream cells actually filter against

read_keep_list <- function(path) {
  # plink-style: one ID per line, or "FID IID" space-separated -- take the
  # last whitespace-separated field either way
  str_trim(read_lines(path)) %>% str_split(" +") %>% sapply(function(x) tail(x, 1))
}

keep_ids_round2 <- read_keep_list(KEEP_LIST_PATH_ROUND2)
keep_ids_round2b <- read_keep_list(KEEP_LIST_PATH_ROUND2B)
keep_ids <- read_keep_list(KEEP_LIST_PATH)

tibble(
  keep_list = c("round2", "round2b", "active (KEEP_LIST_PATH)"),
  n_ids = c(length(keep_ids_round2), length(keep_ids_round2b), length(keep_ids))
)

In [ ]:
pheno_list <- read_tsv("../../docs/phenotype_list.tsv", col_types = cols(.default = "c")) %>%
  filter(concept_id != "UNCONFIRMED") %>%
  slice_head(n = 3)

pheno_list

## Step 1 — concept sanity check

`{CDR}.concept` is public OMOP vocabulary metadata, not participant data —
safe to print in full. Confirms each concept_id exists, is in the expected
domain, and is a standard concept before spending a bigger query on it.

In [ ]:
concept_ids <- paste(pheno_list$concept_id, collapse = ",")
concept_check <- run_query(sprintf("
  SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept
  FROM {CDR}.concept
  WHERE concept_id IN (%s)
", concept_ids))

concept_check

## Step 2 — measurement pull funnel

For each phenotype: total measurement rows for its concept_id, distinct
persons, and the NA rate on `value_as_number` — all aggregate counts, no
person-level output.

In [ ]:
pull_funnel_row <- function(pheno_row) {
  stats <- run_query(sprintf("
    SELECT
      COUNT(*) AS n_rows,
      COUNT(DISTINCT person_id) AS n_persons,
      COUNTIF(value_as_number IS NULL) AS n_null_value
    FROM {CDR}.measurement
    WHERE measurement_concept_id = %s
  ", pheno_row$concept_id))

  stats %>% mutate(phenotype_name = pheno_row$phenotype_name, .before = 1)
}

pull_funnel <- bind_rows(lapply(seq_len(nrow(pheno_list)), function(i) pull_funnel_row(pheno_list[i, ])))
pull_funnel

## Step 3 — keep-list filtering + sex/age join

Most-recent-value-per-person, joined to age and `sex_at_birth_concept_id`
(the AoU-specific column, distinct from `gender_concept_id`) — same shape
`pull_phenotype()` returns in `residualize_phenotypes.ipynb`. Reports the
funnel from raw pull down to the keep-list intersection, plus the
sex_at_birth category breakdown (aggregate counts only).

In [ ]:
REFERENCE_DATE <- "2024-01-01"   # arbitrary fixed date -- adjust to match the CDR version's actual data cutoff
stopifnot(REFERENCE_DATE != "PLACEHOLDER")   # unset fails as an invalid DATE literal deep inside the BigQuery
                                              # job, surfaced by aou_sql() as an opaque "did not result in a table" error

pull_and_filter <- function(pheno_row) {
  # DATE_DIFF returns INT64; bigrquery collects bare INT64 columns as bit64::integer64,
  # which lm() silently mis-coerces into a degenerate fit rather than erroring -- cast to
  # FLOAT64 in the query so age collects as a plain double
  query <- sprintf("
    WITH demographics AS (
      SELECT
        person_id,
        CAST(DATE_DIFF(DATE '%s', DATE(birth_datetime), YEAR) AS FLOAT64) AS age,
        CASE
          WHEN sex_at_birth_concept_id = 45878463 THEN 'Female'
          WHEN sex_at_birth_concept_id = 45880669 THEN 'Male'
          ELSE 'Other'
        END AS sex_at_birth
      FROM {CDR}.person
    ),
    measurements AS (
      SELECT
        person_id,
        value_as_number AS phenotype,
        ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY measurement_date DESC) AS rn
      FROM {CDR}.measurement
      WHERE measurement_concept_id = %s
        AND value_as_number IS NOT NULL
    )
    SELECT d.person_id, m.phenotype, d.age, d.sex_at_birth
    FROM demographics d
    INNER JOIN measurements m ON d.person_id = m.person_id
    WHERE m.rn = 1
  ", REFERENCE_DATE, pheno_row$concept_id)

  df <- run_query(query) %>%
    mutate(person_id = as.character(person_id))

  df_filtered <- df %>% filter(person_id %in% keep_ids)

  list(name = pheno_row$phenotype_name, df = df, df_filtered = df_filtered)
}

pulls <- lapply(seq_len(nrow(pheno_list)), function(i) pull_and_filter(pheno_list[i, ]))
names(pulls) <- pheno_list$phenotype_name

filter_funnel <- bind_rows(lapply(pulls, function(p) {
  tibble(
    phenotype_name = p$name,
    n_before_keep_list = nrow(p$df),
    n_after_keep_list = nrow(p$df_filtered)
  )
}))
filter_funnel

In [ ]:
sex_breakdown <- bind_rows(lapply(pulls, function(p) {
  p$df_filtered %>% count(sex_at_birth) %>% mutate(phenotype_name = p$name, .before = 1)
}))
sex_breakdown

## Step 4 — value range sanity check

Aggregate summary stats per phenotype (min/median/mean/max/SD, NA rate) on
the keep-list-filtered values — a quick physiological-plausibility check
before trusting the pull, not a full distribution.

In [ ]:
value_summary <- bind_rows(lapply(pulls, function(p) {
  x <- p$df_filtered$phenotype
  tibble(
    phenotype_name = p$name,
    n = sum(!is.na(x)),
    min = min(x, na.rm = TRUE),
    median = median(x, na.rm = TRUE),
    mean = mean(x, na.rm = TRUE),
    max = max(x, na.rm = TRUE),
    sd = sd(x, na.rm = TRUE)
  )
}))
value_summary

## Step 5 — mock residualization (base + base_pcs covariate sets)

Runs `residualize_lib.R`'s real `residualize_phenotype()` — the same
function `residualize_phenotypes.ipynb` uses — on the phenotypes just
pulled. `base` (`age` only) runs on the keep-list-filtered pull from Step 3;
`base_pcs` (`age` + round 2b's `PC1..PC5`) joins in round 2b's PC
projections via `inner_join`, which implicitly restricts that run to
round-2b-passing individuals regardless of which keep-list `KEEP_LIST_PATH`
points to. The residualization math itself runs on real AoU values, not
synthetic data like `test_residualize_fake_data.ipynb`. Confirms the
pipeline's core step behaves sanely (retained N, R²) on real data before
trusting it on the full phenotype list and covariate-set cross product.
Diagnostics only, no `.pheno` files written.

In [ ]:
pcs <- read_table(PC_COVARIATE_PATH, col_types = cols(.default = "d", IID = "c")) %>%
  rename(person_id = IID)
pc_cols <- paste0("PC", 1:5)   # top 5 only -- beyond that isn't considered informative for this cohort
covariate_sets <- build_covariate_sets(pc_cols)

mock_residualization <- bind_rows(lapply(pulls, function(p) {
  base <- residualize_phenotype(p$df_filtered, "phenotype", "sex_at_birth", covariate_sets$base)

  df_pcs <- p$df %>% inner_join(pcs, by = "person_id")
  base_pcs <- residualize_phenotype(df_pcs, "phenotype", "sex_at_birth", covariate_sets$base_pcs)

  bind_rows(
    tibble(phenotype_name = p$name, covariate_set = "base",
           n_input = base$n_input, n_retained = base$n_retained, r_squared = base$r_squared),
    tibble(phenotype_name = p$name, covariate_set = "base_pcs",
           n_input = base_pcs$n_input, n_retained = base_pcs$n_retained, r_squared = base_pcs$r_squared)
  )
}))
mock_residualization

## Step 6 — model diagnostics: loadings, encoding, distributions

Deeper look than Step 5's retained-N/R² summary: coefficient tables (how
each phenotype loads onto age, each PC, and the SES variables), how
covariates actually get encoded going into `lm()`, and the raw/transformed
phenotype distributions. Everything below is model-level output
(coefficients, encoding metadata, aggregate distributions/quantiles) —
never a person-level row.

### Pull SES data

Same `zip3_ses_map` join `pull_covariates()` uses in
`residualize_phenotypes.ipynb`, with the same `FLOAT64` casts from the
integer64 fix above. Not phenotype-specific, so pulled once.

In [ ]:
ses_query <- "
  SELECT
    o.person_id,
    o.zip3,
    CAST(z.median_income AS FLOAT64) AS median_income,
    CAST(z.fraction_poverty AS FLOAT64) AS poverty,
    CAST(z.deprivation_index AS FLOAT64) AS deprivation_index
  FROM (
    SELECT person_id, CAST(SUBSTR(value_as_string, 1, 3) AS INT64) AS zip3
    FROM {CDR}.observation
    WHERE STRPOS(value_as_string, '*') > 0
  ) o
  INNER JOIN {CDR}.zip3_ses_map z ON o.zip3 = z.zip3
"

ses <- run_query(ses_query) %>%
  mutate(person_id = as.character(person_id), zip3 = as.character(zip3))   # zip3 is a factor
                                                                            # level label, not numeric -- cast
                                                                            # in R same as pull_covariates()
nrow(ses)

### Fit `base_pcs_ses` and pull out coefficients

Fits `age + PC1..PC5 + median_income + poverty + deprivation_index`
directly with `lm()` (not `residualize_phenotype()`, which returns
diagnostics but not the fit object) so the coefficients themselves are
inspectable. Joins are `inner_join`, so this implicitly restricts to
people with both PCs (round 2b passers) and a masked zip3/SES record.

In [ ]:
tidy_lm <- function(fit) {
  s <- summary(fit)$coefficients
  tibble(
    term = rownames(s),
    estimate = s[, "Estimate"],
    std_error = s[, "Std. Error"],
    t_value = s[, "t value"],
    p_value = s[, "Pr(>|t|)"]
  )
}

fits <- lapply(pulls, function(p) {
  df <- p$df %>% inner_join(pcs, by = "person_id") %>% inner_join(ses, by = "person_id")
  formula <- as.formula(paste("phenotype ~", paste(covariate_sets$base_pcs_ses, collapse = " + ")))
  lm(formula, data = df, na.action = na.exclude)
})
names(fits) <- names(pulls)

sapply(fits, function(f) sum(!is.na(residuals(f))))   # N actually used per phenotype's fit

In [ ]:
coef_table <- bind_rows(lapply(names(fits), function(name) {
  tidy_lm(fits[[name]]) %>% mutate(phenotype_name = name, .before = 1)
}))

# age + SES loadings -- the PC loadings are in the next cell, kept separate since
# there are 20 of them
coef_table %>% filter(term %in% c("age", "median_income", "poverty", "deprivation_index"))

In [ ]:
coef_table %>% filter(grepl("^PC", term))

### Covariate encoding check

Confirms `age`/PCs/SES collected as plain numeric doubles (the integer64
fix above) rather than `bit64::integer64`, and that plink2's
`variance-standardize` scoring actually left the PCs at roughly mean-0/
SD-1. `sex_at_birth` is **not** a regression covariate here — per
`residualize_lib.R`, it's the stratification variable for the separate
within-sex standardization step (step 3 in the module docstring), so
there's no `lm()` factor/contrast encoding for it to check; what's worth
confirming instead is that its values are the expected small category set.

In [ ]:
sapply(pulls[[1]]$df["age"], class)
sapply(pcs[pc_cols], class)
sapply(ses[c("median_income", "poverty", "deprivation_index")], class)

# variance-standardize scoring should leave these close to mean 0, SD 1
sapply(pcs[pc_cols], function(x) c(mean = mean(x, na.rm = TRUE), sd = sd(x, na.rm = TRUE)))

In [ ]:
sex_breakdown_all <- bind_rows(lapply(names(pulls), function(name) {
  pulls[[name]]$df_filtered %>% count(sex_at_birth) %>% mutate(phenotype_name = name, .before = 1)
}))
sex_breakdown_all

### zip3 as a factor covariate

`build_covariate_sets()` also defines `base_pcs_zip3`/`base_pcs_zip3_ses`,
using the 3-digit zip as a categorical covariate — not exercised anywhere
above. `lm()` one-hot-encodes a character column automatically (treatment
contrasts, reference level = alphabetically first), so fitting
`base_pcs_zip3_ses` produces one dummy term per zip3 level minus one. With
~800 possible 3-digit US zip codes, printing every dummy's coefficient by
default would be a lot of noise — summarized below instead; `zip3_fits` has
the full `lm` objects if you want to inspect specific levels.

In [ ]:
# aggregate zip3 frequency -- geographic distribution, not person-level
ses %>% count(zip3, sort = TRUE) %>% head(10)

In [ ]:
zip3_fits <- lapply(pulls, function(p) {
  df <- p$df %>% inner_join(pcs, by = "person_id") %>% inner_join(ses, by = "person_id")
  formula <- as.formula(paste("phenotype ~", paste(covariate_sets$base_pcs_zip3_ses, collapse = " + ")))
  lm(formula, data = df, na.action = na.exclude)
})
names(zip3_fits) <- names(pulls)

zip3_fit_summary <- bind_rows(lapply(names(zip3_fits), function(name) {
  fit <- zip3_fits[[name]]
  s <- summary(fit)$coefficients
  zip3_rows <- grepl("^zip3", rownames(s))
  tibble(
    phenotype_name = name,
    n_zip3_levels_in_data = length(unique(fit$model$zip3)),
    n_zip3_dummy_terms = sum(zip3_rows),
    n_zip3_dummies_sig_p05 = sum(zip3_rows & s[, "Pr(>|t|)"] < 0.05, na.rm = TRUE),
    r_squared = summary(fit)$r.squared,
    n_used = nrow(fit$model)
  )
}))
zip3_fit_summary

### Phenotype distributions

Histograms (raw), a skewness table (raw vs. `inverse_normal_transform()`,
reusing `residualize_lib.R`'s real functions rather than reimplementing),
quantiles, and a by-sex boxplot — all aggregate/visual, no person-level
values printed.

In [ ]:
for (name in names(pulls)) {
  hist(pulls[[name]]$df_filtered$phenotype, breaks = 40, main = paste(name, "-- raw"), xlab = name)
}

In [ ]:
skew_table <- bind_rows(lapply(names(pulls), function(name) {
  x <- pulls[[name]]$df_filtered$phenotype
  bind_rows(
    skew_summary(x, "raw"),
    skew_summary(inverse_normal_transform(x), "invnorm")
  ) %>% mutate(phenotype = name, .before = 1)
}))
skew_table

In [ ]:
quantile_table <- bind_rows(lapply(names(pulls), function(name) {
  x <- pulls[[name]]$df_filtered$phenotype
  q <- quantile(x, probs = c(0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99), na.rm = TRUE)
  tibble(phenotype_name = name, quantile = names(q), value = as.numeric(q))
}))
quantile_table

In [ ]:
for (name in names(pulls)) {
  df <- pulls[[name]]$df_filtered
  boxplot(phenotype ~ sex_at_birth, data = df, main = paste(name, "by sex_at_birth"), ylab = name)
}

## Summary

If `concept_check` matched the expected concepts, `filter_funnel` shows a
sane retention rate against the keep-list, `value_summary`'s ranges look
physiologically plausible (e.g. height in cm, not inches or meters),
`mock_residualization` ran without error with a plausible R², and Step 6's
`coef_table`/`zip3_fit_summary`/encoding checks/distributions all look sane
— the query, filtering, and residualization logic is validated and
`residualize_phenotypes.ipynb` is safe to run on the full phenotype list
and covariate-set cross product.

Still needs the real workbench before that: confirming `WORKSPACE_BUCKET`
matches the actual mounted resource name, and that `REFERENCE_DATE` is
appropriate for the CDR version in use.